In [ ]:
!pip install -q japanize-matplotlib


<a id="top"></a>

# 【PTCG AI Battle】Replay Archetype Analysis(環境デッキ分析)（日本語版）

Kaggle The Pokémon Company - PTCG AI Battle Challenge の上位対戦履歴から daily replay JSON、または保存済みCSVを読み込み、カード名を日本語で表示しながら環境分析を行うNotebookです。私が作成したコードを、chatgptを使い、公開用に編集しました。


## 英語版(English Version)について
このコードの別Versionに、英語版のノートブックがあります。 There's an English version of the notebook for this code.

## 準備
https://www.kaggle.com/datasets/kaggle/pokemon-tcg-ai-battle-episodes-index を参照し、Add Inputから各日のデータを追加してください。

## このNotebookでできること

- 指定日付・期間の replay JSON を読み込む
- 保存済みCSVにも最新のarchetype分類を再適用する
- カード採用率、archetype使用率、同一60枚デッキ勝率を集計する
- 各archetypeの勝率上位1〜5位デッキをCSV / TXT / PYとして保存する
- archetype別の相性表とヒートマップを作成する
- `その他/不明` に残ったデッキから新archetype候補を探す
- 必要に応じて、勝率上位デッキに紐づく replay action trace を保存する

## 公開ノートブックの更新方針

この公開Notebookを毎日1回実行し、最新のリプレイデータを用いて分析を行っていきます。

分析結果は、リプレイデータの追加や環境の変化に応じて日々変動します。

Version名は以下の形式で管理しています。

* JP-2026-06-26
* EN-2026-06-26
* JP-2026-06-27
* EN-2026-06-27

本Notebookは継続的に更新されるリプレイデータから、環境デッキの分布、勝率、採用カード、デッキ相性などを追跡することを目的としています。

また、環境上位に新たに登場したアーキタイプや、使用率・勝率が大きく変化したデッキを記録し、メタゲームの変遷を継続的に観察することを目指しています。

## コピー＆エディットする際に変更する場所

基本的に **[1. 設定](#section-1)** のコードセルだけ変更すれば回せるようにしてあります。特に `ANALYSIS_MODE`、`DATE_START`、`DATE_END`、`DEBUG_MODE` を確認してください。設定セルの直後に、各設定変数の意味を表で表示しています。    

archtypeの分類を変更したい場合はclassify_deck関数を変更してください。この関数は4章の2つ目のセルにあり、viewerではhideせずに表示しています。    

<a id="toc"></a>

## 目次

- [1. 設定](#section-1)  
  入力データ、分析期間、出力先、debug条件、集計条件を指定します。主な出力は、設定変数の解説表です。
- [2. カードマスタと共通関数](#section-2)  
  日本語カード名、カード種別、ACE SPEC判定、進化ライン、同名別IDの扱いを準備します。主な出力は、同名・別IDカードの確認表です。
- [3. 入力ファイルの解決](#section-3)  
  指定日付・期間に対応する daily episode dataset を探し、読み込むJSONを決定します。主な出力は、日付・入力フォルダ確認表です。
- [4. replay JSON / 保存済みCSVの読み込み](#section-4)  
  raw JSONまたは保存済みCSVを読み込み、`matches`、`decklists`、`logs` の3表にそろえます。
- [5. 集計](#section-5)  
  カード採用率、archetype使用率、行動候補別勝率などを集計します。主な出力は、`archetype_usage`、`card_usage`、`strong_actions` です。
- [6. デッキランキングと保存](#section-6)  
  同一60枚デッキ単位で勝率を集計し、各archetypeの上位デッキを保存します。主な出力は、top deck CSV / TXT / PYです。
- [6.5 replay action trace 出力（任意）](#section-6-5)  
  勝率上位デッキと実リプレイ行動を紐づけて保存します。行動ログ分析やBC/RL用データ作成に使います。
- [7. archetype相性](#section-7)  
  自分のarchetype × 相手archetypeの勝率・対戦数を集計します。主な出力は、matchup summaryとheatmapです。
- [8. 新archetype候補](#section-8)  
  `その他/不明` に残ったデッキから、勝率と採用カードに基づいて新archetype候補を抽出します。
- [9. 可視化と出力一覧](#section-9)  
  主要グラフを保存し、作成ファイルの用途を一覧化します。主な出力は、figuresと`output_file_descriptions.csv`です。


<a id="section-1"></a>

## 1. 設定

[目次へ戻る](#toc)

この章では、入力データ、分析期間、出力先、debug条件、集計条件を指定します。  
初見の場合は、まず `ANALYSIS_MODE`、`DATE_START`、`DATE_END`、`DEBUG_MODE` だけ確認すれば動かしやすいです。

- `ANALYSIS_MODE = "raw"`: replay JSON から新しく抽出します。
- `ANALYSIS_MODE = "saved"`: 保存済みCSVを読み込みます。
- `ANALYSIS_MODE = "auto"`: raw JSONがあればraw、なければsavedへ自動で切り替えます。


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import display, Markdown
import itertools
import json
import re
import hashlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================
# 入力設定
# =========================
ANALYSIS_MODE = "raw"  # "raw", "saved", "auto"

EPISODES_BASE_DIR = Path("/kaggle/input/datasets/organizations/kaggle")
DAILY_EPISODE_DIR_PREFIX = "pokemon-tcg-ai-battle-episodes-"

DATE_START = "2026-07-08"
DATE_END = "2026-07-08"
DATE_LIST = []  # 例: ["2026-06-18", "2026-06-20"]

MANUAL_DATA_DIRS = []  # 直接フォルダを指定したい場合のみ使う
SAVED_DIR = Path("/kaggle/input/notebooks/smallpond/ptcg-analysis-japanese")
JP_CARD_PATH = Path("/kaggle/input/competitions/pokemon-tcg-ai-battle/JP_Card_Data.csv")

# =========================
# 出力設定
# =========================
OUT_DIR = Path("/kaggle/working/ptcg_analysis_jp_public_v15")
DIRS = {
    "raw": OUT_DIR / "01_raw_extraction",
    "analysis": OUT_DIR / "02_reanalysis",
    "deck": OUT_DIR / "03_deck_ranking",
    "candidate": OUT_DIR / "04_archetype_discovery",
    "fig": OUT_DIR / "figures",
}
DIRS["ready"] = DIRS["deck"] / "ready_to_use_top_decks"
# v15: BC/RL用に、勝率上位デッキと実リプレイ行動を紐づけて保存する。
DIRS["bc"] = OUT_DIR / "05_behavior_cloning_data"
DIRS["deck_action"] = OUT_DIR / "06_deck_action_data"

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# =========================
# Debug設定
# =========================
DEBUG_MODE = False
DEBUG_MAX_REPLAY_FILES = 100
DEBUG_RANDOM_SAMPLE = False
DEBUG_RANDOM_SEED = 42
MAX_REPLAY_FILES = DEBUG_MAX_REPLAY_FILES if DEBUG_MODE else None

# =========================
# 集計条件
# =========================
MIN_ACTION_GAMES = 10
MIN_DECK_GAMES_FOR_RANKING = 5
TOP_N_GLOBAL_DECKS = 10
TOP_N_BY_ARCHETYPE = 5
MIN_ARCHETYPE_GAMES_FOR_WINRATE_PLOT = 1
MIN_MATCHUP_GAMES_FOR_DISPLAY = 5
MIN_CANDIDATE_WINRATE = 0.50
MIN_CANDIDATE_GAMES = 3

# =========================
# v15: Behavior Cloning用 action trace 出力設定
# =========================
SAVE_TOP_DECK_ACTION_TRACES = False
TOP_DECK_ACTION_WINNER_ONLY_MAIN = True
TOP_DECK_ACTION_WINNER_WEIGHT = 3.0
TOP_DECK_ACTION_LOSER_WEIGHT = 0.5
MAX_ACTION_TRACE_FILES = MAX_REPLAY_FILES  # Debug時はMAX_REPLAY_FILESに連動。Noneなら全件。
MAX_ACTION_TRACE_ROWS = None               # 大規模化しすぎる場合は 100000 などにする。


# =========================
# 汎用カード名パターン
# =========================
# グッズ・スタジアムはカード種別から自動判定し、ACE SPEC以外は汎用扱い。
GENERIC_CARD_PATTERNS = [
    "基本", "エネルギー", "ボスの指令", "リーリエの決心", "ポケパッド",
    "ハイパーボール", "なかよしポフィン", "夜のタンカ", "ポケモンいれかえ",
    "ふしぎなアメ", "ジャッジマン", "博士の研究",
    "エネルギー転送", "エネルギー回収", "エネルギーつけかえ",
]

print("ANALYSIS_MODE:", ANALYSIS_MODE)
print("OUT_DIR:", OUT_DIR)
print("DEBUG_MODE:", DEBUG_MODE, "MAX_REPLAY_FILES:", MAX_REPLAY_FILES)


In [ ]:

# =========================
# 設定変数の解説表
# =========================
# 初見の人は、この表を見ながら上の設定値だけを変更すれば使えます。

CONFIG_GUIDE = pd.DataFrame([
    {
        "カテゴリ": "入力",
        "変数名": "ANALYSIS_MODE",
        "現在の値": ANALYSIS_MODE,
        "説明": "分析モード。raw=Replay JSONから抽出、saved=保存済みCSVを再分析、auto=rawがあればraw、なければsaved。",
        "よく変更するか": "はい",
    },
    {
        "カテゴリ": "入力",
        "変数名": "EPISODES_BASE_DIR",
        "現在の値": str(EPISODES_BASE_DIR),
        "説明": "daily replay datasetが置かれているKaggle inputの親フォルダ。通常は変更不要です。",
        "よく変更するか": "いいえ",
    },
    {
        "カテゴリ": "入力",
        "変数名": "DAILY_EPISODE_DIR_PREFIX",
        "現在の値": DAILY_EPISODE_DIR_PREFIX,
        "説明": "daily replay datasetフォルダ名の接頭辞。日付を後ろにつけて入力フォルダを探します。",
        "よく変更するか": "いいえ",
    },
    {
        "カテゴリ": "期間",
        "変数名": "DATE_START / DATE_END",
        "現在の値": f"{DATE_START} 〜 {DATE_END}",
        "説明": "分析する日付範囲。連続した期間を分析したい場合に指定します。",
        "よく変更するか": "はい",
    },
    {
        "カテゴリ": "期間",
        "変数名": "DATE_LIST",
        "現在の値": str(DATE_LIST),
        "説明": "飛び飛びの日付だけを分析したい場合に使います。空リストなら DATE_START〜DATE_END を使います。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "入力",
        "変数名": "MANUAL_DATA_DIRS",
        "現在の値": str(MANUAL_DATA_DIRS),
        "説明": "入力フォルダを手動で指定したい場合に使います。通常は空でOKです。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "入力",
        "変数名": "SAVED_DIR",
        "現在の値": str(SAVED_DIR),
        "説明": "ANALYSIS_MODE='saved' のときに、保存済みCSVを探すフォルダです。",
        "よく変更するか": "saved利用時のみ",
    },
    {
        "カテゴリ": "入力",
        "変数名": "JP_CARD_PATH",
        "現在の値": str(JP_CARD_PATH),
        "説明": "カードIDと日本語カード名を対応させるカードマスタCSVです。",
        "よく変更するか": "いいえ",
    },
    {
        "カテゴリ": "出力",
        "変数名": "OUT_DIR / DIRS",
        "現在の値": str(OUT_DIR),
        "説明": "集計CSV、デッキリスト、画像、action traceなどの保存先です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Debug",
        "変数名": "DEBUG_MODE",
        "現在の値": DEBUG_MODE,
        "説明": "Trueにすると処理するJSON数を制限して試運転できます。最初はTrueで確認するのもおすすめです。",
        "よく変更するか": "はい",
    },
    {
        "カテゴリ": "Debug",
        "変数名": "DEBUG_MAX_REPLAY_FILES",
        "現在の値": DEBUG_MAX_REPLAY_FILES,
        "説明": "DEBUG_MODE=True のときに読み込むreplay JSONの最大数です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Debug",
        "変数名": "DEBUG_RANDOM_SAMPLE / DEBUG_RANDOM_SEED",
        "現在の値": f"{DEBUG_RANDOM_SAMPLE}, seed={DEBUG_RANDOM_SEED}",
        "説明": "debug時に先頭からではなくランダムサンプルするか、またその乱数seedです。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "集計条件",
        "変数名": "MIN_ACTION_GAMES",
        "現在の値": MIN_ACTION_GAMES,
        "説明": "行動候補別勝率に表示するための最小player-game数です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "集計条件",
        "変数名": "MIN_DECK_GAMES_FOR_RANKING",
        "現在の値": MIN_DECK_GAMES_FOR_RANKING,
        "説明": "同一60枚デッキランキングに載せるための最小player-game数です。",
        "よく変更するか": "はい",
    },
    {
        "カテゴリ": "集計条件",
        "変数名": "TOP_N_GLOBAL_DECKS / TOP_N_BY_ARCHETYPE",
        "現在の値": f"overall={TOP_N_GLOBAL_DECKS}, by_archetype={TOP_N_BY_ARCHETYPE}",
        "説明": "全体ランキングとarchetype別ランキングで表示・保存する上位件数です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "集計条件",
        "変数名": "MIN_MATCHUP_GAMES_FOR_DISPLAY",
        "現在の値": MIN_MATCHUP_GAMES_FOR_DISPLAY,
        "説明": "archetype相性表で信頼できる対面として扱う最小対戦数です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "新archetype候補",
        "変数名": "MIN_CANDIDATE_WINRATE / MIN_CANDIDATE_GAMES",
        "現在の値": f"winrate>={MIN_CANDIDATE_WINRATE}, games>={MIN_CANDIDATE_GAMES}",
        "説明": "その他/不明デッキから候補を抽出するための勝率・件数条件です。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Action trace",
        "変数名": "SAVE_TOP_DECK_ACTION_TRACES",
        "現在の値": SAVE_TOP_DECK_ACTION_TRACES,
        "説明": "勝率上位デッキの実リプレイ行動ログを保存するかどうかです。分析だけならFalseでも構いません。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Action trace",
        "変数名": "TOP_DECK_ACTION_WINNER_ONLY_MAIN",
        "現在の値": TOP_DECK_ACTION_WINNER_ONLY_MAIN,
        "説明": "主に勝者側の行動を学習用データとして重視するかどうかです。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Action trace",
        "変数名": "TOP_DECK_ACTION_WINNER_WEIGHT / TOP_DECK_ACTION_LOSER_WEIGHT",
        "現在の値": f"winner={TOP_DECK_ACTION_WINNER_WEIGHT}, loser={TOP_DECK_ACTION_LOSER_WEIGHT}",
        "説明": "勝者・敗者のaction traceに付与する重みです。",
        "よく変更するか": "必要に応じて",
    },
    {
        "カテゴリ": "Action trace",
        "変数名": "MAX_ACTION_TRACE_FILES / MAX_ACTION_TRACE_ROWS",
        "現在の値": f"files={MAX_ACTION_TRACE_FILES}, rows={MAX_ACTION_TRACE_ROWS}",
        "説明": "action trace保存処理が大きくなりすぎる場合の上限制御です。Noneなら制限しません。",
        "よく変更するか": "必要に応じて",
    },
])

display(Markdown("### 設定変数の解説"))
display(Markdown("各変数の意味と、初見の人が変更するべきかをまとめた表です。"))
display(CONFIG_GUIDE)


<a id="section-2"></a>

## 2. カードマスタと共通関数

[目次へ戻る](#toc)

この章では、カードIDと日本語カード名の対応、カード種別、ACE SPEC判定、進化ライン、同名別IDの扱いをまとめます。以降の集計でカード名を読みやすく表示するための土台です。


In [ ]:
def save_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    return path



def display_table(title, df, description="", max_rows=None):
    """
    DataFrameを表示するときに、必ず説明を付けるための共通関数。
    max_rowsを指定すると先頭行だけを表示する。
    """
    display(Markdown(f"### {title}"))

    shape_text = ""
    if isinstance(df, pd.DataFrame):
        shape_text = f"\n\n- shape: `{df.shape[0]} rows × {df.shape[1]} columns`"

    if description:
        display(Markdown(description + shape_text))
    elif shape_text:
        display(Markdown(shape_text))

    if df is None:
        display(Markdown("`None`"))
        return

    if isinstance(df, pd.DataFrame) and len(df) == 0:
        display(Markdown("条件を満たす行はありません。"))
        display(df)
        return

    if max_rows is not None and isinstance(df, pd.DataFrame):
        with pd.option_context("display.max_rows",max_rows, "display.max_columns", None):
            display(df)
    else:
        display(df)


def write_markdown_file(text, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path


def normalize_bool_series(s):
    if len(s) == 0:
        return s
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False, "yes": True, "no": False,
    }).fillna(False)


def sanitize_filename(text, max_len=80):
    text = re.sub(r"[\\/:*?\"<>|]+", "_", str(text))
    text = re.sub(r"\s+", "_", text).strip("_")
    return text[:max_len]


def load_jp_card_master(path=JP_CARD_PATH):
    jp = pd.read_csv(path)
    cols = {
        "id": "カード ID",
        "name": "カード名",
        "stage": "ポケモンの進化の段階/エネルギー・トレーナーズの種類",
        "rule": "ルール",
        "category": "カテゴリ",
        "pre": "進化前",
    }
    missing = [c for c in cols.values() if c not in jp.columns]
    if missing:
        raise ValueError(f"JP_Card_Data.csv missing columns: {missing}")

    jp = jp.copy()
    jp[cols["id"]] = pd.to_numeric(jp[cols["id"]], errors="coerce").astype("Int64")
    id_master = jp.dropna(subset=[cols["id"]]).sort_values(cols["id"]).drop_duplicates(cols["id"]).copy()
    name_master = jp.dropna(subset=[cols["name"]]).sort_values(cols["id"]).drop_duplicates(cols["name"]).copy()

    def dict_by_id(col):
        return dict(zip(id_master[cols["id"]].astype(int), id_master[col].fillna("").astype(str)))

    def dict_by_name(col):
        return dict(zip(name_master[cols["name"]].astype(str), name_master[col].fillna("").astype(str)))

    dup = (
        id_master.groupby(cols["name"], dropna=False)
        .agg(
            num_card_ids=(cols["id"], "nunique"),
            card_ids=(cols["id"], lambda x: ", ".join(map(str, sorted(set(int(v) for v in x.dropna()))))),
            stages=(cols["stage"], lambda x: " / ".join(sorted(set(str(v) for v in x.dropna())))),
            rules=(cols["rule"], lambda x: " / ".join(sorted(set(str(v) for v in x.dropna() if str(v) != "nan")))),
        )
        .reset_index()
    )
    dup = dup[dup["num_card_ids"] >= 2].sort_values(["num_card_ids", cols["name"]], ascending=[False, True])

    children = defaultdict(set)
    for _, r in jp.iterrows():
        name = str(r.get(cols["name"], "") or "")
        pre = str(r.get(cols["pre"], "") or "")
        if name and pre and pre != "nan":
            children[pre].add(name)

    return {
        "raw": jp,
        "id_master": id_master,
        "id_to_name": dict_by_id(cols["name"]),
        "id_to_stage": dict_by_id(cols["stage"]),
        "id_to_rule": dict_by_id(cols["rule"]),
        "id_to_category": dict_by_id(cols["category"]),
        "id_to_pre": dict_by_id(cols["pre"]),
        "name_to_stage": dict_by_name(cols["stage"]),
        "name_to_rule": dict_by_name(cols["rule"]),
        "name_to_category": dict_by_name(cols["category"]),
        "name_to_pre": dict_by_name(cols["pre"]),
        "duplicate_name_summary": dup,
        "duplicate_card_names": set(dup[cols["name"]].astype(str)),
        "evolution_children": children,
        "cols": cols,
    }


JP = load_jp_card_master(JP_CARD_PATH)
JP_DF = JP["raw"]
DUPLICATE_CARD_NAMES = JP["duplicate_card_names"]
PRE_EVOLUTION_NAMES = set(JP_DF["進化前"].dropna().astype(str).loc[lambda x: x.str.len() > 0].tolist())

save_csv(JP["duplicate_name_summary"], OUT_DIR / "duplicate_card_name_master.csv")
display_table(
    "同名・別IDカード一覧",
    JP["duplicate_name_summary"],
    "同じ日本語カード名で複数のカードIDを持つカードです。デッキ表示では、効果違いを区別するために `カード名[ID:xxx]` と表示します。",
    max_rows=200,
)


def to_jp_name(card_id, english_name=None):
    if card_id is None or pd.isna(card_id):
        return english_name
    try:
        return JP["id_to_name"].get(int(card_id), english_name or f"UNKNOWN_{int(card_id)}")
    except Exception:
        return english_name or f"UNKNOWN_{card_id}"


def card_meta(card_id=None, card_name=None):
    if card_id is not None and pd.notna(card_id):
        try:
            cid = int(card_id)
            return {
                "jp_stage": JP["id_to_stage"].get(cid, ""),
                "jp_rule": JP["id_to_rule"].get(cid, ""),
                "jp_category": JP["id_to_category"].get(cid, ""),
                "jp_evolves_from": JP["id_to_pre"].get(cid, ""),
            }
        except Exception:
            pass

    name = str(card_name)
    return {
        "jp_stage": JP["name_to_stage"].get(name, ""),
        "jp_rule": JP["name_to_rule"].get(name, ""),
        "jp_category": JP["name_to_category"].get(name, ""),
        "jp_evolves_from": JP["name_to_pre"].get(name, ""),
    }


def is_ace_spec(name=None, rule=None):
    return "ACE SPEC" in f"{name or ''} {rule or ''}".upper()


def is_goods_or_stadium(stage):
    s = str(stage)
    return ("グッズ" in s) or ("スタジアム" in s)


def format_card_label(card_id, card_name, always_show_id=False):
    name = str(card_name)
    try:
        cid = int(card_id)
    except Exception:
        return name
    if always_show_id or name in DUPLICATE_CARD_NAMES:
        return f"{name}[ID:{cid}]"
    return name


def terminal_evolution_names(card_name, max_depth=20):
    start = str(card_name)
    terminals, stack, seen = set(), [(start, 0)], set()
    while stack:
        cur, depth = stack.pop()
        if cur in seen or depth > max_depth:
            continue
        seen.add(cur)
        children = sorted(JP["evolution_children"].get(cur, []))
        if not children:
            terminals.add(cur)
        else:
            stack.extend((c, depth + 1) for c in children)
    return sorted(terminals or {start})


def final_evolution_score(name):
    meta = card_meta(card_name=name)
    stage = meta["jp_stage"]
    score = 0
    score += 40 if "2進化" in stage else 25 if "1進化" in stage else 5 if ("たね" in stage or "基本" in stage) else 0
    score += 20 if str(name).startswith("メガ") else 0
    score += 10 if "ex" in str(name) else 0
    score += 5 if str(name) not in PRE_EVOLUTION_NAMES else 0
    return score


def final_evolution_name(card_name):
    terminals = terminal_evolution_names(card_name)
    return sorted(terminals, key=lambda x: (final_evolution_score(x), x), reverse=True)[0]


def add_card_master_columns(df):
    if len(df) == 0 or "card_id" not in df.columns:
        return df

    out = df.copy()
    out["card_id_int"] = pd.to_numeric(out["card_id"], errors="coerce").astype("Int64")
    for col in ["stage", "rule", "category", "pre"]:
        out[f"jp_{'evolves_from' if col == 'pre' else col}"] = out["card_id_int"].map(
            lambda x, col=col: JP[f"id_to_{col}"].get(int(x), "") if pd.notna(x) else ""
        )

    if "card_name" in out.columns:
        out["is_ace_spec"] = out.apply(lambda r: is_ace_spec(r.get("card_name"), r.get("jp_rule")), axis=1)
        out["is_goods_or_stadium"] = out["jp_stage"].apply(is_goods_or_stadium)
        out["is_pre_evolution"] = out["card_name"].astype(str).isin(PRE_EVOLUTION_NAMES)
        out["evolution_representative"] = out["card_name"].apply(final_evolution_name)
        out["evolution_family"] = out["evolution_representative"]
        out["is_duplicate_card_name"] = out["card_name"].astype(str).isin(DUPLICATE_CARD_NAMES)
        out["card_label"] = out.apply(lambda r: format_card_label(r.get("card_id_int"), r.get("card_name")), axis=1)
        out["card_label_with_id"] = out.apply(lambda r: format_card_label(r.get("card_id_int"), r.get("card_name"), True), axis=1)
    return out


def is_generic_candidate_card(row_or_name):
    if isinstance(row_or_name, pd.Series):
        name, stage, rule = str(row_or_name.get("card_name", "")), str(row_or_name.get("jp_stage", "")), str(row_or_name.get("jp_rule", ""))
    else:
        name = str(row_or_name)
        meta = card_meta(card_name=name)
        stage, rule = meta["jp_stage"], meta["jp_rule"]

    if is_goods_or_stadium(stage) and not is_ace_spec(name, rule):
        return True
    return any(p in name for p in GENERIC_CARD_PATTERNS)


<a id="section-3"></a>

## 3. 入力ファイルの解決

[目次へ戻る](#toc)

指定した日付・期間に対応する daily replay dataset を探し、処理対象のJSONファイルを決定します。debug時は、ここで読み込むJSON数を制限します。


In [ ]:
def resolve_dates(date_start=DATE_START, date_end=DATE_END, date_list=DATE_LIST):
    if date_list:
        return sorted(set(pd.to_datetime(pd.Series(date_list)).dt.strftime("%Y-%m-%d").tolist()))
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(date_start, date_end, freq="D")]


def collect_episode_dirs():
    rows, dirs = [], []

    if MANUAL_DATA_DIRS:
        for p in map(Path, MANUAL_DATA_DIRS):
            m = re.search(r"(\d{4}-\d{2}-\d{2})", str(p))
            n = len(list(p.rglob("*.json"))) if p.exists() else 0
            rows.append({"episode_date": m.group(1) if m else None, "path": str(p), "exists": p.exists(), "json_count": n, "source": "manual"})
            if p.exists():
                dirs.append(p)
        return dirs, pd.DataFrame(rows)

    for d in resolve_dates():
        p = EPISODES_BASE_DIR / f"{DAILY_EPISODE_DIR_PREFIX}{d}"
        n = len(list(p.rglob("*.json"))) if p.exists() else 0
        rows.append({"episode_date": d, "path": str(p), "exists": p.exists(), "json_count": n, "source": "date_range"})
        if p.exists():
            dirs.append(p)
    return dirs, pd.DataFrame(rows)


def select_json_files(files, max_files=MAX_REPLAY_FILES, random_sample=DEBUG_RANDOM_SAMPLE, seed=DEBUG_RANDOM_SEED):
    files = sorted(set(map(Path, files)))
    if max_files is None or len(files) <= max_files:
        return files
    if random_sample:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(files), size=max_files, replace=False)
        return [files[i] for i in sorted(idx)]
    return files[:max_files]


data_dirs, date_selection_summary = collect_episode_dirs()
json_files_all = [p for d in data_dirs for p in sorted(Path(d).rglob("*.json"))]
json_files = select_json_files(json_files_all)

save_csv(date_selection_summary, DIRS["raw"] / "date_selection_summary.csv")
display_table(
    "日付・入力フォルダの確認",
    date_selection_summary,
    "指定した日付範囲に対応するdaily replay datasetが見つかったかを確認する表です。`exists=True` かつ `json_count > 0` の日付が分析対象になります。",
)

print("json files all:", len(json_files_all))
print("json files selected:", len(json_files))
#for p in json_files[:10]:
    #print(p)


<a id="section-4"></a>

## 4. replay JSON / 保存済みCSVの読み込み

[目次へ戻る](#toc)

raw replay JSONから抽出する場合でも、保存済みCSVを読み込む場合でも、最終的に `matches` / `decklists` / `logs` の3表へそろえます。


In [ ]:
ACTION_TYPES = {"Play", "Attach", "Attack", "Evolve", "Switch", "Retreat", "Ability", "UseAbility"}

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_visualize(data):
    for step in data.get("steps", []):
        if isinstance(step, list):
            for agent_step in step:
                if isinstance(agent_step, dict) and isinstance(agent_step.get("visualize"), list):
                    return agent_step["visualize"]
        elif isinstance(step, dict) and isinstance(step.get("visualize"), list):
            return step["visualize"]
    return []


def infer_date(path):
    m = re.search(r"pokemon-tcg-ai-battle-episodes-(\d{4}-\d{2}-\d{2})", str(path))
    return m.group(1) if m else None


def episode_id(data, path=None):
    info = data.get("info", {})
    for k in ["EpisodeId", "episodeId", "id"]:
        if k in info:
            return info[k]
    if "id" in data:
        return data["id"]
    if path:
        m = re.search(r"(\d{5,})", Path(path).stem)
        return int(m.group(1)) if m else str(path)
    return None


def teams(data):
    info = data.get("info", {})
    return info.get("TeamNames") or info.get("teamNames") or []


def winner(data):
    rewards = data.get("rewards") or []
    if not rewards:
        return None
    mx = max(rewards)
    return rewards.index(mx) if rewards.count(mx) == 1 else None


def iter_cards(obj):
    if isinstance(obj, dict):
        if isinstance(obj.get("id"), int) and ("name" in obj or "serial" in obj):
            yield obj
        for v in obj.values():
            yield from iter_cards(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_cards(v)


def card_maps(data):
    id_to_name, serial_to_card = {}, {}
    for c in iter_cards(data):
        if c.get("id") is not None and c.get("name"):
            id_to_name[c["id"]] = c["name"]
        if "serial" in c:
            serial_to_card[c["serial"]] = c
    return id_to_name, serial_to_card


def card_from_log(log, id_to_name, serial_to_card, id_key="cardId", serial_key="serial"):
    serial = log.get(serial_key)
    if serial in serial_to_card:
        c = serial_to_card[serial]
        return c.get("id"), c.get("name") or id_to_name.get(c.get("id"))
    cid = log.get(id_key)
    return cid, id_to_name.get(cid, f"UNKNOWN_{cid}" if cid is not None else None)

In [ ]:
def classify_deck(name_counts, id_counts=None):
    names, id_counts = set(name_counts), Counter(id_counts or {})
    has = lambda n: n in names
    has_id = lambda cid: id_counts.get(cid, 0) > 0
    count_names = lambda xs: sum(int(name_counts.get(x, 0)) for x in xs)
    prefix_count = lambda p: sum(int(v) for k, v in name_counts.items() if str(k).startswith(p))

    # 細かい分類を先に判定する。
    if has("ナンジャモのハラバリーex"):
        return "ナンジャモのハラバリーex", "ナンジャモのハラバリーex"
    if has("シロナのガブリアスex"):
        return "シロナのガブリアスex", "シロナのガブリアスex"
    if has("Nのゾロアークex"):
        return "Nのゾロアークex","Nのゾロアークex"
    if has("マリィのオーロンゲex"):
        return "マリィのオーロンゲex", "マリィのオーロンゲex"
    if has("トドゼルガ"):
        return "トドゼルガ","トドゼルガ"
        
    if has("フーディン") and has("ノココッチ"):
        return "フーディン + ノココッチ", "フーディン, ノココッチ"
    if has("キュワワー") and has("シャンデラ"):
        return "キュワワー + シャンデラ", "キュワワー, シャンデラ"
    if has("イワパレス") and has("ヒビキのバクフーン"):
        return "イワパレス + ヒビキのバクフーン", "イワパレス, ヒビキのバクフーン"
    if (has_id(93) or has("カミッチュ")) and has("バチンキー") and has("お祭り会場"):
        return "お祭りデッキ", "カミッチュ[ID:93], バチンキー, お祭り会場"
    if has("ヒビキのバクフーン") and has("ノココッチ"):
        return "ヒビキのバクフーン + ノココッチ", "ヒビキのバクフーン, ノココッチ"
    if has("ヒビキのバクフーン") and has("ドラメシヤ"):
        return "ヒビキのバクフーン + ドラメシヤ", "ヒビキのバクフーン, ドラメシヤ"
    if has("ピカチュウex") and has("マリルリ"):
        return "ピカチュウex + マリルリ","ピカチュウex, マリルリ"
    if has("ヤドキング") and has("夜のアカデミー"):
        return "ヤドキング ＋ 夜のアカデミー","ヤドキング, 夜のアカデミー"

    hop_pokemon = [
        "ホップのボクレー", "ホップのオーロット", "ホップのウッウ", "ホップのカビゴン",
        "ホップのココガラ", "ホップのアオガラス", "ホップのウールー", "ホップのバイウールー",
        "ホップのザシアンex", "ホップのサダイジャ", "ホップのスナヘビ",
    ]
    if count_names(hop_pokemon) >= 4 and prefix_count("ホップの") >= 8:
        return "ホップデッキ", "ホップのポケモン中心"

    tags = []
    if {"メガルカリオex", "リオル"} & names:
        tags.append("メガルカリオex")
    if {"ソルロック", "ルナトーン"} <= names:
        tags.append("ソルロック/ルナトーン")
    if {"ドラパルトex", "ドロンチ", "ドラメシヤ"} & names:
        tags.append("ドラパルトex")
    if {"イワパレス", "イシズマイ"} & names:
        tags.append("イワパレス")
    if "イイネイヌ" in names or "イイネイヌex" in names:
        tags.append("イイネイヌ")
    if {"ガメノデス", "カメテテ"} & names:
        tags.append("ガメノデス")
    if {"メガサーナイトex", "サーナイトex", "キルリア", "ラルトス"} & names:
        tags.append("サーナイト")
    if {"メガリザードンXex", "メガリザードンYex", "ヒトカゲ", "リザード"} & names:
        tags.append("メガリザードン")
    if {"ブリジュラスex","ジュラルドン"} & names:
        tags.append("ブリジュラスex")
    if any(str(n).startswith("ロケット団") for n in names):
        tags.append("ロケット団")

    MEGA_ARCHETYPE_RULES_JP = [
    # Composite Mega decks
    ("メガスターミーex + メガユキメノコex", {"メガスターミーex", "メガユキメノコex"}),
    ("メガスターミーex + メガミミロップex", {"メガスターミーex", "メガミミロップex"}),
    ("メガユキメノコex + メガミミロップex", {"メガユキメノコex", "メガミミロップex"}),]

    for label, required_cards in MEGA_ARCHETYPE_RULES_JP:
        if required_cards.issubset(names):
            if label not in tags:
                tags.append(label)
    # Fallback: any remaining Mega ex cards not covered by explicit rules
    for m in sorted(n for n in names if isinstance(n, str) and n.startswith("メガ") and "ex" in n):
        if m not in tags:
            tags.append(m)

    if not tags:
        return "その他/不明", "その他/不明"
    if "メガルカリオex" in tags and "ソルロック/ルナトーン" in tags:
        return "メガルカリオex + ソルロック/ルナトーン", ", ".join(tags)
    if "ドラパルトex" in tags:
        return "ドラパルトex", ", ".join(tags)
    if "イイネイヌ" in tags and "ガメノデス" in tags:
        return "イイネイヌ + ガメノデス", ", ".join(tags)
    if "イワパレス" in tags:
        return "イワパレス", ", ".join(tags)
    return tags[0], ", ".join(tags)

In [ ]:
def extract_decklists(data, path):
    viz = get_visualize(data)
    if not viz:
        return pd.DataFrame()

    eid, ep_date, team_names, rewards, win = str(episode_id(data, path)), infer_date(path), teams(data), data.get("rewards") or [], winner(data)
    players = (viz[0].get("current") or {}).get("players") or []
    rows = []

    for pi, player in enumerate(players):
        counter = Counter()
        for c in player.get("deck") or []:
            if isinstance(c, dict):
                counter[(c.get("id"), c.get("name"), to_jp_name(c.get("id"), c.get("name")))] += 1

        name_counts, id_counts = Counter(), Counter()
        for (cid, en, jp), n in counter.items():
            name_counts[jp or en or f"UNKNOWN_{cid}"] += n
            if cid is not None:
                id_counts[int(cid)] += n
        archetype, tags = classify_deck(name_counts, id_counts)

        for (cid, en, jp), n in counter.most_common():
            rows.append({
                "episode_id": eid, "episode_date": ep_date, "source_file": str(path),
                "player": pi, "team": team_names[pi] if pi < len(team_names) else None,
                "reward": rewards[pi] if pi < len(rewards) else None, "won": pi == win,
                "archetype": archetype, "archetype_tags": tags,
                "card_id": cid, "card_name": jp, "card_name_jp": jp, "card_name_en": en, "count": n,
                **card_meta(cid),
            })

    return add_card_master_columns(pd.DataFrame(rows))


def extract_logs(data, path):
    viz = get_visualize(data)
    if not viz:
        return pd.DataFrame()

    eid, ep_date, team_names, rewards, win = str(episode_id(data, path)), infer_date(path), teams(data), data.get("rewards") or [], winner(data)
    id_to_name, serial_to_card = card_maps(data)
    rows = []

    for vi, x in enumerate(viz):
        current, obs_current = x.get("current") or {}, (x.get("obs") or {}).get("current") or {}
        for li, log in enumerate(x.get("logs") or []):
            if not isinstance(log, dict):
                continue

            cid, en = card_from_log(log, id_to_name, serial_to_card)
            tid, ten = card_from_log(log, id_to_name, serial_to_card, "cardIdTarget", "serialTarget")
            player = log.get("playerIndex")
            jp, tjp = to_jp_name(cid, en), to_jp_name(tid, ten)

            rows.append({
                "episode_id": eid, "episode_date": ep_date, "source_file": str(path),
                "viz_index": vi, "log_index": li,
                "turn_before": obs_current.get("turn"), "turn_after": current.get("turn"),
                "actor_obs": obs_current.get("yourIndex"),
                "player": player, "team": team_names[player] if isinstance(player, int) and player < len(team_names) else None,
                "reward": rewards[player] if isinstance(player, int) and player < len(rewards) else None,
                "won": player == win if isinstance(player, int) and win is not None else None,
                "log_type": log.get("type"),
                "card_id": cid, "card_name": jp, "card_name_jp": jp, "card_name_en": en,
                "target_card_id": tid, "target_card_name": tjp, "target_card_name_jp": tjp, "target_card_name_en": ten,
                "attack_id": log.get("attackId"), "value": log.get("value"),
                "from_area": log.get("fromArea"), "to_area": log.get("toArea"),
                "raw": json.dumps(log, ensure_ascii=False),
                **card_meta(cid),
            })
    return add_card_master_columns(pd.DataFrame(rows))


def extract_matches(data, path):
    viz = get_visualize(data)
    if not viz:
        return pd.DataFrame()

    eid, ep_date, team_names, rewards, statuses, win = str(episode_id(data, path)), infer_date(path), teams(data), data.get("rewards") or [], data.get("statuses") or [], winner(data)
    deck_df = extract_decklists(data, path)
    arch = {int(r.player): (r.archetype, r.archetype_tags) for r in deck_df.drop_duplicates(["episode_id", "player"]).itertuples()} if len(deck_df) else {}

    final = viz[-1].get("current", {}) or {}
    players = final.get("players") or []
    rows = []

    for pi in range(max(2, len(players), len(rewards), len(team_names))):
        p = players[pi] if pi < len(players) else {}
        a, tags = arch.get(pi, (None, None))
        prize_left = len(p.get("prize") or [])
        rows.append({
            "episode_id": eid, "episode_date": ep_date, "source_file": str(path),
            "player": pi, "team": team_names[pi] if pi < len(team_names) else None,
            "reward": rewards[pi] if pi < len(rewards) else None,
            "status": statuses[pi] if pi < len(statuses) else None,
            "won": pi == win, "winner_player": win,
            "archetype": a, "archetype_tags": tags,
            "final_turn": final.get("turn"), "num_viz_steps": len(viz),
            "final_prize_remaining": prize_left, "final_prize_taken": 6 - prize_left,
            "final_deck_count": p.get("deckCount"), "final_hand_count": p.get("handCount"),
        })
    return pd.DataFrame(rows)


def analyze_replay_files(files):
    decklists, logs, matches, bad = [], [], [], []
    for i, p in enumerate(files):
        if i % 50 == 0:
            print(f"processing {i}/{len(files)}")
        try:
            data = load_json(p)
            if not get_visualize(data):
                bad.append((str(p), "no visualize"))
                continue
            decklists.append(extract_decklists(data, p))
            logs.append(extract_logs(data, p))
            matches.append(extract_matches(data, p))
        except Exception as e:
            bad.append((str(p), repr(e)))

    tables = {
        "decklists": pd.concat(decklists, ignore_index=True) if decklists else pd.DataFrame(),
        "event_logs": pd.concat(logs, ignore_index=True) if logs else pd.DataFrame(),
        "matches": pd.concat(matches, ignore_index=True) if matches else pd.DataFrame(),
        "bad_files": pd.DataFrame(bad, columns=["file", "error"]),
    }

    for name, df in tables.items():
        save_csv(df, DIRS["raw"] / f"{name}.csv")
    return tables


def find_csv(saved_dir, name):
    direct = Path(saved_dir) / f"{name}.csv"
    if direct.exists():
        return direct
    cands = list(Path(saved_dir).rglob(f"{name}.csv")) if Path(saved_dir).exists() else []
    return cands[0] if cands else None


def load_saved_tables(saved_dir):
    names = ["matches", "decklists", "event_logs", "bad_files"]
    out = {}
    for n in names:
        p = find_csv(saved_dir, n)
        out[n] = pd.read_csv(p) if p else pd.DataFrame()
        print("loaded" if p else "missing", n, p or "")
    return out


def reclassify(decklists, matches=None):
    if len(decklists) == 0:
        return decklists, matches

    out = decklists.copy()
    for (eid, player), g in out.groupby(["episode_id", "player"]):
        name_counts, id_counts = Counter(), Counter()
        for r in g.itertuples():
            n = int(getattr(r, "count", 0) or 0)
            name_counts[str(r.card_name)] += n
            if pd.notna(r.card_id):
                id_counts[int(r.card_id)] += n
        a, tags = classify_deck(name_counts, id_counts)
        mask = (out["episode_id"].astype(str) == str(eid)) & (out["player"] == player)
        out.loc[mask, ["archetype", "archetype_tags"]] = [a, tags]

    if matches is not None and len(matches):
        m = out[["episode_id", "player", "archetype", "archetype_tags"]].drop_duplicates(["episode_id", "player"])
        matches = matches.drop(columns=["archetype", "archetype_tags"], errors="ignore").merge(m, on=["episode_id", "player"], how="left")
    return out, matches


def prepare_tables(source):
    matches = source.get("matches", pd.DataFrame()).copy()
    decklists = source.get("decklists", pd.DataFrame()).copy()
    logs = source.get("event_logs", pd.DataFrame()).copy()

    for df in [matches, decklists, logs]:
        if len(df) and "episode_id" in df.columns:
            df["episode_id"] = df["episode_id"].astype(str)
        if len(df) and "won" in df.columns:
            df["won"] = normalize_bool_series(df["won"])

    if len(decklists):
        decklists["card_id"] = pd.to_numeric(decklists["card_id"], errors="coerce").astype("Int64")
        decklists["count"] = pd.to_numeric(decklists["count"], errors="coerce").fillna(0).astype(int)
        decklists = add_card_master_columns(decklists)

    if len(logs) and "card_id" in logs.columns:
        logs["card_id"] = pd.to_numeric(logs["card_id"], errors="coerce").astype("Int64")
        logs = add_card_master_columns(logs)

    decklists, matches = reclassify(decklists, matches)
    return matches, decklists, logs


use_saved = ANALYSIS_MODE == "saved" or (ANALYSIS_MODE == "auto" and find_csv(SAVED_DIR, "matches"))
source = load_saved_tables(SAVED_DIR) if use_saved else analyze_replay_files(json_files)
matches, decklists, logs = prepare_tables(source)

print("matches:", matches.shape)
print("decklists:", decklists.shape)
print("logs:", logs.shape)

display_table(
    "matches の先頭行",
    matches,
    "1行が `episode_id × player` に対応します。勝敗、archetype、最終ターン、残りサイドなどを確認できます。",
    max_rows=5,
)

display_table(
    "decklists の先頭行",
    decklists,
    "1行が `episode_id × player × card` に対応します。各プレイヤーの60枚デッキをカード単位で確認できます。",
    max_rows=5,
)

display_table(
    "logs の先頭行",
    logs,
    "1行がreplay中のログイベントです。カード使用、進化、攻撃、移動などの行動を確認できます。",
    max_rows=5,
)

<a id="section-5"></a>

## 5. 集計

[目次へ戻る](#toc)

カード、archetype、行動、対面を集計します。個別カード固定リストに依存しすぎないよう、カード種別や進化ラインも利用します。


In [ ]:
def rebuild_basic_tables(matches, decklists, logs):
    card_usage = (
        decklists.groupby(["card_id", "card_name"], dropna=False)
        .agg(player_decks=("episode_id", "count"), total_copies=("count", "sum"), avg_copies=("count", "mean"), wins=("won", "sum"))
        .reset_index()
        if len(decklists) else pd.DataFrame()
    )
    if len(card_usage):
        card_usage["winrate"] = card_usage["wins"] / card_usage["player_decks"]
        for c in ["card_label", "card_name_en", "jp_stage", "jp_rule", "jp_category"]:
            if c in decklists.columns:
                card_usage[c] = card_usage["card_id"].map(decklists.drop_duplicates("card_id").set_index("card_id")[c].to_dict())
        card_usage = card_usage.sort_values(["player_decks", "winrate"], ascending=[False, False])

    archetype_usage = (
        matches.groupby("archetype", dropna=False)
        .agg(player_decks=("episode_id", "count"), wins=("won", "sum"), avg_reward=("reward", "mean"))
        .reset_index()
        if len(matches) and "archetype" in matches.columns else pd.DataFrame()
    )
    if len(archetype_usage):
        archetype_usage["winrate"] = archetype_usage["wins"] / archetype_usage["player_decks"]
        archetype_usage["winrate_pct"] = (archetype_usage["winrate"] * 100).round(1)
        archetype_usage = archetype_usage.sort_values(["player_decks", "winrate"], ascending=[False, False])

    if len(logs):
        action_ep = (
            logs[logs["log_type"].isin(ACTION_TYPES) & logs["card_name"].notna()]
            .groupby(["episode_id", "player", "log_type", "card_name"], dropna=False)
            .agg(times=("log_type", "count"), won=("won", "max"))
            .reset_index()
        )
        strong_actions = (
            action_ep.groupby(["log_type", "card_name"], dropna=False)
            .agg(player_games=("episode_id", "count"), wins=("won", "sum"), avg_times=("times", "mean"))
            .reset_index()
        )
        strong_actions["winrate_when_used"] = strong_actions["wins"] / strong_actions["player_games"]
        strong_actions = strong_actions[strong_actions["player_games"] >= MIN_ACTION_GAMES].sort_values(["winrate_when_used", "player_games"], ascending=[False, False])
    else:
        strong_actions = pd.DataFrame()


    return {
        "matches": matches, "decklists": decklists, "event_logs": logs,
        "card_usage": card_usage, "archetype_usage": archetype_usage,
        "strong_actions": strong_actions,
    }


tables = rebuild_basic_tables(matches, decklists, logs)
for name, df in tables.items():
    save_csv(df, DIRS["analysis"] / f"{name}.csv")

display_table(
    "archetype_usage: archetype別の使用数・勝率",
    tables["archetype_usage"],
    "各archetypeが何回使われたか、何勝したか、勝率がどれくらいかをまとめた表です。環境で多いデッキタイプを確認します。",
    max_rows=30,
)

display_table(
    "card_usage: カード別の採用数・勝率",
    tables["card_usage"],
    "各カードが何個のplayer-deckに採用されたか、合計採用枚数、採用デッキの勝率をまとめた表です。",
    max_rows=30,
)

display_table(
    "strong_actions: 行動候補別の勝率",
    tables["strong_actions"],
    f"カード使用・進化・攻撃などの行動について、その行動が観測されたplayer-gameの勝率をまとめた表です。`player_games >= {MIN_ACTION_GAMES}` の行動だけを表示します。",
    max_rows=30,
)


<a id="section-6"></a>

## 6. デッキランキングと保存

[目次へ戻る](#toc)

同一60枚デッキを `deck_signature` で識別し、勝率上位を集計します。各archetypeの上位デッキは、Markdownで読みやすく表示し、CSV / TXT / PYとして保存します。


In [ ]:
def make_deck_signature(g):
    tmp = g[["card_id", "card_name", "count"]].copy()
    tmp["cid"] = pd.to_numeric(tmp["card_id"], errors="coerce")
    tmp["count"] = pd.to_numeric(tmp["count"], errors="coerce").fillna(0).astype(int)
    return "|".join(
        f"{'id:' + str(int(r.cid)) if pd.notna(r.cid) else 'name:' + str(r.card_name)}:{int(r.count)}"
        for r in tmp.sort_values(["cid", "card_name"]).itertuples()
    )


def deck_detail(g, top_n=None, core=False, sep=" / "):
    tmp = add_card_master_columns(g.copy())
    if core:
        tmp = tmp[~tmp.apply(is_generic_candidate_card, axis=1)].copy()
        if len(tmp) == 0:
            tmp = add_card_master_columns(g.copy())

    tmp = tmp.groupby(["card_id", "card_name", "card_label"], dropna=False).agg(count=("count", "sum")).reset_index()
    tmp["cid"] = pd.to_numeric(tmp["card_id"], errors="coerce")
    tmp = tmp.sort_values(["count", "card_name", "cid"], ascending=[False, True, True])
    if top_n:
        tmp = tmp.head(top_n)
    return sep.join(f"{r.card_label}×{int(r.count)}" for r in tmp.itertuples())


def build_deck_instances(decklists):
    rows = []
    for (eid, player), g in decklists.groupby(["episode_id", "player"]):
        rows.append({
            "episode_id": str(eid),
            "player": player,
            "team": g["team"].iloc[0] if "team" in g else None,
            "archetype": g["archetype"].iloc[0] if "archetype" in g else None,
            "archetype_tags": g["archetype_tags"].iloc[0] if "archetype_tags" in g else None,
            "won": bool(g["won"].iloc[0]) if "won" in g else None,
            "reward": g["reward"].iloc[0] if "reward" in g else np.nan,
            "deck_signature": make_deck_signature(g),
            "deck_detail_japanese": deck_detail(g),
            "deck_core_japanese": deck_detail(g, top_n=10, core=True),
            "num_cards_total": int(g["count"].sum()),
            "num_unique_cards": int(g["card_id"].nunique()),
        })
    return pd.DataFrame(rows)


def deck_performance(deck_instances):
    perf = (
        deck_instances.groupby(["archetype", "deck_signature"], dropna=False)
        .agg(
            player_games=("episode_id", "count"), wins=("won", "sum"), avg_reward=("reward", "mean"),
            num_unique_teams=("team", "nunique"), archetype_tags=("archetype_tags", "first"),
            deck_detail_japanese=("deck_detail_japanese", "first"), deck_core_japanese=("deck_core_japanese", "first"),
            example_episode_id=("episode_id", "first"), example_player=("player", "first"),
            num_cards_total=("num_cards_total", "first"), num_unique_cards=("num_unique_cards", "first"),
        )
        .reset_index()
    )
    perf["winrate"] = perf["wins"] / perf["player_games"]
    perf["winrate_pct"] = (perf["winrate"] * 100).round(1)
    return perf.sort_values(["winrate", "player_games", "avg_reward"], ascending=[False, False, False])


def top_deck_display_columns(df):
    cols = [
        "rank_overall", "rank_in_archetype", "archetype", "player_games", "wins", "winrate_pct", "avg_reward",
        "num_unique_teams", "num_unique_cards", "deck_core_japanese", "deck_detail_japanese",
        "example_episode_id", "example_player",
    ]
    return [c for c in cols if c in df.columns]


def make_top_decks_markdown(df, title, rank_col=None, group_by_archetype=False):
    lines = [f"# {title}", ""]
    lines.append(f"対象: 同一60枚デッキの対戦回数 `player_games >= {MIN_DECK_GAMES_FOR_RANKING}` のもの。")
    lines.append("")

    if len(df) == 0:
        lines.append("条件を満たすデッキはありません。")
        return "\n".join(lines)

    if group_by_archetype:
        for arch, g in df.sort_values(["archetype", rank_col]).groupby("archetype", dropna=False):
            lines.append(f"## {arch}")
            for r in g.itertuples():
                rank = int(getattr(r, rank_col))
                lines.append(f"### #{rank} 勝率 {float(r.winrate_pct):.1f}% ({int(r.wins)}/{int(r.player_games)})")
                lines.append(f"- 平均reward: {float(r.avg_reward):.3f}")
                lines.append(f"- コア: {r.deck_core_japanese}")
                lines.append(f"- 60枚詳細: {r.deck_detail_japanese}")
                lines.append(f"- 代表episode/player: `{r.example_episode_id}` / `{r.example_player}`")
                lines.append("")
    else:
        for r in df.sort_values(rank_col).itertuples():
            rank = int(getattr(r, rank_col))
            lines.append(f"## #{rank} {r.archetype} — 勝率 {float(r.winrate_pct):.1f}% ({int(r.wins)}/{int(r.player_games)})")
            lines.append(f"- 平均reward: {float(r.avg_reward):.3f}")
            lines.append(f"- コア: {r.deck_core_japanese}")
            lines.append(f"- 60枚詳細: {r.deck_detail_japanese}")
            lines.append(f"- 代表episode/player: `{r.example_episode_id}` / `{r.example_player}`")
            lines.append("")

    return "\n".join(lines)


def display_top_decks_markdown(df, title, rank_col=None, group_by_archetype=False, save_path=None):
    md_text = make_top_decks_markdown(df, title, rank_col=rank_col, group_by_archetype=group_by_archetype)
    display(Markdown(md_text))
    if save_path is not None:
        write_markdown_file(md_text, save_path)


deck_instances = build_deck_instances(decklists)
exact_deck_perf = deck_performance(deck_instances)
save_csv(deck_instances, DIRS["deck"] / "deck_instances.csv")
save_csv(exact_deck_perf, DIRS["deck"] / "exact_deck_performance_all.csv")

top10 = exact_deck_perf[exact_deck_perf["player_games"] >= MIN_DECK_GAMES_FOR_RANKING].head(TOP_N_GLOBAL_DECKS).copy()
top10.insert(0, "rank_overall", range(1, len(top10) + 1))
save_csv(top10, DIRS["deck"] / "top10_decks_overall_min5games.csv")

display_table(
    "全archetype横断: 勝率上位10デッキ",
    top10[top_deck_display_columns(top10)],
    f"同一60枚デッキ単位で集計した勝率上位デッキです。`player_games >= {MIN_DECK_GAMES_FOR_RANKING}` のデッキだけを対象にしています。下にMarkdown形式でも表示します。",
)
display_top_decks_markdown(
    top10,
    "全archetype横断: 勝率上位10デッキ",
    rank_col="rank_overall",
    group_by_archetype=False,
    save_path=DIRS["deck"] / "top10_decks_overall_min5games.md",
)


all_archetypes = sorted(set(tables["archetype_usage"]["archetype"].dropna().astype(str))) if len(tables["archetype_usage"]) else []

eligible = exact_deck_perf[exact_deck_perf["player_games"] >= MIN_DECK_GAMES_FOR_RANKING].copy()
top_by_arch = (
    eligible.sort_values(["archetype", "winrate", "player_games", "avg_reward"], ascending=[True, False, False, False])
    .groupby("archetype", dropna=False)
    .head(TOP_N_BY_ARCHETYPE)
    .copy()
)
if len(top_by_arch):
    top_by_arch["rank_in_archetype"] = top_by_arch.groupby("archetype", dropna=False).cumcount() + 1
else:
    top_by_arch["rank_in_archetype"] = pd.Series(dtype="int")

save_csv(top_by_arch, DIRS["deck"] / "top5_decks_by_archetype_min5games.csv")

display_table(
    "archetype別: 勝率上位1〜5位デッキ",
    top_by_arch[top_deck_display_columns(top_by_arch)],
    f"各archetypeについて、勝率上位1〜5位の同一60枚デッキを表示します。`player_games >= {MIN_DECK_GAMES_FOR_RANKING}` のデッキだけを対象にしています。下にarchetype別Markdown形式でも表示します。",
)
display_top_decks_markdown(
    top_by_arch,
    "archetype別: 勝率上位1〜5位デッキ",
    rank_col="rank_in_archetype",
    group_by_archetype=True,
    save_path=DIRS["deck"] / "top5_decks_by_archetype_min5games.md",
)

# 条件を満たすデッキがないarchetypeを明示する。
missing_arch = sorted(set(all_archetypes) - set(top_by_arch["archetype"].astype(str))) if len(top_by_arch) else all_archetypes
if missing_arch:
    display(Markdown(
        "### 条件を満たすデッキがないarchetype\n\n"
        + f"必要条件: `player_games >= {MIN_DECK_GAMES_FOR_RANKING}`\n\n"
        + "\n".join(f"- {x}" for x in missing_arch)
    ))


def deck_rows_from_top(row):
    g = decklists[(decklists["episode_id"].astype(str) == str(row["example_episode_id"])) & (decklists["player"] == row["example_player"])].copy()
    return add_card_master_columns(g).sort_values(["count", "card_name", "card_id"], ascending=[False, True, True])


def save_ready_decks(top_df):
    index_rows, long_rows = [], []
    for _, row in top_df.sort_values(["archetype", "rank_in_archetype"]).iterrows():
        arch, rank = str(row["archetype"]), int(row["rank_in_archetype"])
        wr, games, wins = float(row["winrate_pct"]), int(row["player_games"]), int(row["wins"])
        g = deck_rows_from_top(row)
        if len(g) == 0:
            continue

        base = f"{sanitize_filename(arch)}__rank_{rank:02d}__winrate_{wr:.1f}__games_{games}".replace(".", "_")
        csv_path, txt_path, py_path = DIRS["ready"] / f"{base}.csv", DIRS["ready"] / f"{base}.txt", DIRS["ready"] / f"{base}.py"

        export_cols = [c for c in ["card_id", "card_label", "card_name", "card_name_jp", "card_name_en", "count", "jp_stage", "jp_rule", "jp_category"] if c in g]
        out = g[export_cols].copy()
        # v15: 後段のBC/RL notebookで、デッキと行動ログを確実に結合できるように deck_signature を同梱する。
        out.insert(0, "deck_signature", row["deck_signature"])
        out.insert(1, "archetype", arch)
        out.insert(2, "rank_in_archetype", rank)
        out.insert(3, "winrate_pct", wr)
        out.insert(4, "wins", wins)
        out.insert(5, "player_games", games)
        save_csv(out, csv_path)

        txt = [f"archetype: {arch}", f"rank_in_archetype: {rank}", f"winrate: {wr:.1f}% ({wins}/{games})", "", "decklist:"]
        txt += [f"{r.card_label} x {int(r.count)}" for r in g.itertuples()]
        txt_path.write_text("\n".join(txt), encoding="utf-8")

        repeated_ids, counts = [], {}
        for r in g.itertuples():
            cid, cnt = int(r.card_id), int(r.count)
            repeated_ids += [cid] * cnt
            counts[cid] = cnt

        py_path.write_text(
            f"# archetype: {arch}\n# rank_in_archetype: {rank}\n# winrate: {wr:.1f}% ({wins}/{games})\n\n"
            f"DECK_CARD_IDS = {repeated_ids!r}\n\nDECK_COUNTS = {counts!r}\n",
            encoding="utf-8",
        )

        index_rows.append({
            "archetype": arch, "rank_in_archetype": rank, "winrate_pct": wr,
            "wins": wins, "player_games": games,
            "csv_path": str(csv_path), "txt_path": str(txt_path), "py_path": str(py_path),
            "deck_signature": row["deck_signature"],
            "example_episode_id": row["example_episode_id"], "example_player": row["example_player"],
        })
        long_rows.extend(out.to_dict("records"))

    index_df, long_df = pd.DataFrame(index_rows), pd.DataFrame(long_rows)
    save_csv(index_df, DIRS["ready"] / "top_deck_files_index.csv")
    save_csv(long_df, DIRS["deck"] / "top5_decklists_by_archetype_ready_to_use.csv")
    return index_df, long_df


ready_index, ready_long = save_ready_decks(top_by_arch)

display_table(
    "保存した勝率上位デッキファイル一覧",
    ready_index,
    "各archetypeの勝率上位1〜5位デッキについて、CSV/TXT/PYの保存先をまとめた索引です。TXTはコピー用、PYは `DECK_CARD_IDS` / `DECK_COUNTS` として使えます。",
)

display_table(
    "保存した勝率上位デッキのカードリスト一覧",
    ready_long,
    "各archetype上位デッキのカードリストを縦持ちでまとめた表です。1行が1カードに対応します。",
    max_rows=100,
)


<a id="section-6-5"></a>

## 6.5 replay action trace 出力（任意）

[目次へ戻る](#toc)

勝率上位デッキに紐づく実リプレイ行動ログを保存します。後段の分析・学習に使わない場合は、設定セルで `SAVE_TOP_DECK_ACTION_TRACES = False` にしてください。


In [ ]:

# =========================
# v15: 勝率上位デッキに紐づく実リプレイ行動ログを保存
# =========================
# 目的:
# - 後段のモデル学習Notebookで、rule agentの自己生成データだけでなく、daily top episodesに含まれる実行動をBC教師として使う。
# - 勝率上位デッキCSVと同じ deck_signature で action traces を結合できるようにする。
#
# 重要:
# - BCでは「選ばれた行動」だけでなく「その局面の合法手一覧」が必要。
# - そのため observation_json と selected_indices_json を保存する。
# - model notebook側で cg.api.to_observation_class(observation_json) に変換し、state/action featuresを作る。


def json_dumps_safe(x):
    try:
        return json.dumps(x, ensure_ascii=False, separators=(",", ":"))
    except Exception:
        try:
            return json.dumps(str(x), ensure_ascii=False)
        except Exception:
            return "null"


def json_loads_maybe(x):
    if isinstance(x, (dict, list)):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return None
    return None


def normalize_replay_action_indices(action):
    """Kaggle episode の action 表現を selected option index の list[int] に寄せる。"""
    if action is None:
        return []
    if isinstance(action, str):
        s = action.strip()
        try:
            return normalize_replay_action_indices(json.loads(s))
        except Exception:
            vals = []
            for tok in s.replace(",", " ").replace("[", " ").replace("]", " ").split():
                try:
                    vals.append(int(tok))
                except Exception:
                    pass
            return vals
    if isinstance(action, (int, np.integer)):
        return [int(action)]
    if isinstance(action, (list, tuple)):
        vals = []
        for x in action:
            if isinstance(x, (int, np.integer)):
                vals.append(int(x))
            elif isinstance(x, str):
                try:
                    vals.append(int(x))
                except Exception:
                    pass
        return vals
    if isinstance(action, dict):
        for k in ["action", "actions", "selected", "selection", "indices", "index", "value", "response"]:
            if k in action:
                vals = normalize_replay_action_indices(action[k])
                if vals:
                    return vals
    return []


def get_agent_step_observation_dict(agent_step):
    """steps[*][player] から observation dict を取り出す。複数形式を許容する。"""
    if not isinstance(agent_step, dict):
        return None
    for k in ["observation", "obs", "state"]:
        x = agent_step.get(k)
        if isinstance(x, dict):
            return x
    # current/select が直下にある形式にも対応。
    if isinstance(agent_step.get("current"), dict) or isinstance(agent_step.get("select"), dict):
        return agent_step
    return None


def get_agent_step_action_raw(agent_step):
    if not isinstance(agent_step, dict):
        return None
    for k in ["action", "actions", "selected", "selection", "response"]:
        if k in agent_step:
            return agent_step.get(k)
    return None


def iter_agent_steps_for_actions(data):
    """Kaggle episode stepsから、(step_id, player, agent_step) をyieldする。"""
    for si, step in enumerate(data.get("steps") or []):
        if isinstance(step, list):
            for pi, agent_step in enumerate(step):
                if isinstance(agent_step, dict):
                    yield si, pi, agent_step
        elif isinstance(step, dict):
            # actor1人分だけが入っている形式。
            if any(k in step for k in ["observation", "obs", "action", "actions", "current", "select"]):
                yield si, step.get("playerIndex", step.get("player", None)), step
            # players/agents配列に入っている形式。
            for key in ["agents", "players"]:
                if isinstance(step.get(key), list):
                    for pi, agent_step in enumerate(step[key]):
                        if isinstance(agent_step, dict):
                            yield si, pi, agent_step


def nested_get(d, path, default=None):
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def extract_legal_options_raw_from_obs(obs_dict):
    if not isinstance(obs_dict, dict):
        return []
    # よくある形式: observation.select.option
    opt = nested_get(obs_dict, ["select", "option"], None)
    if isinstance(opt, list):
        return opt
    # 一部形式では options/legal_actions など。
    for k in ["option", "options", "legal_options", "legalActions", "legal_actions"]:
        x = obs_dict.get(k)
        if isinstance(x, list):
            return x
    return []


def infer_player_from_observation_dict(obs_dict, fallback=None):
    if isinstance(obs_dict, dict):
        for path in [["current", "yourIndex"], ["select", "playerIndex"], ["playerIndex"], ["player"]]:
            val = nested_get(obs_dict, path, None)
            try:
                if val is not None:
                    return int(val)
            except Exception:
                pass
    try:
        if fallback is not None:
            return int(fallback)
    except Exception:
        pass
    return None


def observation_turn(obs_dict):
    for path in [["current", "turn"], ["turn"]]:
        val = nested_get(obs_dict, path, None)
        try:
            if val is not None:
                return int(val)
        except Exception:
            pass
    return None


def observation_context(obs_dict):
    val = nested_get(obs_dict, ["select", "context"], None)
    return val


def build_top_deck_meta_maps(deck_instances, top_df):
    """episode/player -> deck metadata, top deck signature set を作る。"""
    if deck_instances is None or len(deck_instances) == 0 or top_df is None or len(top_df) == 0:
        return {}, set()
    top_sigs = set(top_df["deck_signature"].dropna().astype(str)) if "deck_signature" in top_df.columns else set()
    meta_cols = [
        "episode_id", "player", "deck_signature", "archetype", "archetype_tags", "won", "reward", "team",
        "deck_core_japanese", "deck_detail_japanese",
    ]
    meta_cols = [c for c in meta_cols if c in deck_instances.columns]
    m = deck_instances[meta_cols].copy()
    m["episode_id"] = m["episode_id"].astype(str)
    out = {}
    for r in m.itertuples(index=False):
        d = r._asdict()
        try:
            key = (str(d["episode_id"]), int(d["player"]))
            out[key] = d
        except Exception:
            pass
    return out, top_sigs


def extract_action_traces_from_episode(data, path, meta_map, top_sigs, winner_only_main=False):
    eid = str(episode_id(data, path))
    ep_date = infer_date(path)
    win = winner(data)
    rewards = data.get("rewards") or []
    rows = []
    stats = Counter()

    for step_id, fallback_player, agent_step in iter_agent_steps_for_actions(data):
        obs_dict = get_agent_step_observation_dict(agent_step)
        action_raw = get_agent_step_action_raw(agent_step)
        if obs_dict is None or action_raw is None:
            stats["missing_obs_or_action"] += 1
            continue

        player = infer_player_from_observation_dict(obs_dict, fallback_player)
        if player is None:
            stats["unknown_player"] += 1
            continue

        key = (eid, int(player))
        if key not in meta_map:
            stats["no_deck_meta"] += 1
            continue
        meta = meta_map[key]
        deck_sig = str(meta.get("deck_signature", ""))
        if top_sigs and deck_sig not in top_sigs:
            stats["not_top_deck"] += 1
            continue

        is_winner_action = (win is not None and int(player) == int(win))
        if winner_only_main and win is not None and not is_winner_action:
            stats["non_winner_skipped"] += 1
            continue

        legal_options = extract_legal_options_raw_from_obs(obs_dict)
        if not legal_options:
            stats["no_legal_options"] += 1
            continue
        selected = normalize_replay_action_indices(action_raw)
        if len(selected) > 20:
            stats["too_many_selected"] += 1
            continue
        selected = [int(i) for i in selected if 0 <= int(i) < len(legal_options)]
        if not selected:
            stats["no_valid_selected"] += 1
            continue

        try:
            reward = float(rewards[int(player)]) if int(player) < len(rewards) else float(meta.get("reward", 0.0) or 0.0)
        except Exception:
            reward = float(meta.get("reward", 0.0) or 0.0)
        sample_weight = TOP_DECK_ACTION_WINNER_WEIGHT if is_winner_action else TOP_DECK_ACTION_LOSER_WEIGHT

        rows.append({
            "action_trace_id": f"{eid}__p{int(player)}__s{int(step_id)}",
            "episode_id": eid,
            "episode_date": ep_date,
            "source_file": str(path),
            "step_id": int(step_id),
            "player": int(player),
            "winner_player": win,
            "is_winner_action": bool(is_winner_action),
            "reward": reward,
            "sample_weight": float(sample_weight),
            "archetype": meta.get("archetype"),
            "archetype_tags": meta.get("archetype_tags"),
            "deck_signature": deck_sig,
            "team": meta.get("team"),
            "deck_core_japanese": meta.get("deck_core_japanese"),
            "turn": observation_turn(obs_dict),
            "select_context": observation_context(obs_dict),
            "option_count": int(len(legal_options)),
            "selected_indices_json": json_dumps_safe(selected),
            "action_raw_json": json_dumps_safe(action_raw),
            "legal_options_json": json_dumps_safe(legal_options),
            "observation_json": json_dumps_safe(obs_dict),
        })
        stats["ok"] += 1

    return rows, stats


def save_table_prefer_parquet(df, path):
    """parquetを優先して保存。環境にpyarrow等がなければpickle/csvにfallback。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception as e:
        pkl = path.with_suffix(".pkl")
        try:
            df.to_pickle(pkl)
            print(f"parquet保存に失敗したためpickle保存: {pkl} ({repr(e)})")
            return pkl
        except Exception:
            csv = path.with_suffix(".csv")
            df.to_csv(csv, index=False)
            print(f"parquet/pickle保存に失敗したためcsv保存: {csv} ({repr(e)})")
            return csv


def collect_and_save_top_deck_action_traces(files, top_df, deck_instances):
    if not SAVE_TOP_DECK_ACTION_TRACES:
        print("SAVE_TOP_DECK_ACTION_TRACES=False のため、action trace保存をスキップします。")
        return pd.DataFrame(), pd.DataFrame()
    if files is None or len(files) == 0:
        print("json filesがないため、action trace保存をスキップします。")
        return pd.DataFrame(), pd.DataFrame()
    if top_df is None or len(top_df) == 0:
        print("top deckがないため、action trace保存をスキップします。")
        return pd.DataFrame(), pd.DataFrame()

    meta_map, top_sigs = build_top_deck_meta_maps(deck_instances, top_df)
    rows, total_stats = [], Counter()
    use_files = select_json_files(files, max_files=MAX_ACTION_TRACE_FILES, random_sample=DEBUG_RANDOM_SAMPLE, seed=DEBUG_RANDOM_SEED)
    print("action trace target files:", len(use_files), "top deck signatures:", len(top_sigs))

    for i, p in enumerate(use_files):
        if i % 100 == 0:
            print(f"action trace extraction {i}/{len(use_files)} rows={len(rows)}")
        try:
            data = load_json(p)
            rr, st = extract_action_traces_from_episode(data, p, meta_map, top_sigs, winner_only_main=False)
            rows.extend(rr)
            total_stats.update(st)
            if MAX_ACTION_TRACE_ROWS is not None and len(rows) >= MAX_ACTION_TRACE_ROWS:
                rows = rows[:MAX_ACTION_TRACE_ROWS]
                break
        except Exception as e:
            total_stats["file_error"] += 1
            if DEBUG_MODE:
                print("action trace error", p, repr(e))

    action_df = pd.DataFrame(rows)
    print("action trace stats:", dict(total_stats))
    print("action trace rows:", action_df.shape)

    if len(action_df) == 0:
        save_csv(pd.DataFrame([dict(total_stats)]), DIRS["bc"] / "action_trace_extraction_stats.csv")
        return action_df, pd.DataFrame()

    winner_df = action_df[action_df["is_winner_action"]].copy()

    all_path = save_table_prefer_parquet(action_df, DIRS["bc"] / "action_traces_top_decks_all.parquet")
    winner_path = save_table_prefer_parquet(winner_df, DIRS["bc"] / "action_traces_top_decks_winner_only.parquet")
    print("saved action traces all:", all_path)
    print("saved action traces winner only:", winner_path)

    # デッキごとの個別action traceも保存する。
    per_deck_rows = []
    for sig, g in action_df.groupby("deck_signature", dropna=False):
        meta = g.iloc[0]
        arch = str(meta.get("archetype", "unknown"))
        stable_sig_hash = hashlib.md5(str(sig).encode("utf-8")).hexdigest()[:10]
        base = f"{sanitize_filename(arch)}__{stable_sig_hash}"
        p = save_table_prefer_parquet(g, DIRS["deck_action"] / f"{base}_actions.parquet")
        per_deck_rows.append({
            "deck_signature": sig,
            "archetype": arch,
            "action_rows": int(len(g)),
            "winner_action_rows": int(g["is_winner_action"].sum()),
            "actions_path": str(p),
        })
    per_deck_index = pd.DataFrame(per_deck_rows).sort_values(["winner_action_rows", "action_rows"], ascending=[False, False])
    save_csv(per_deck_index, DIRS["deck_action"] / "deck_action_files_index.csv")

    # ready_indexにaction trace情報を結合した索引も保存する。
    if "ready_index" in globals() and isinstance(ready_index, pd.DataFrame) and len(ready_index):
        enriched = ready_index.merge(per_deck_index, on=["deck_signature", "archetype"], how="left")
        save_csv(enriched, DIRS["ready"] / "top_deck_files_index_with_actions.csv")
    else:
        enriched = pd.DataFrame()

    summary = (
        action_df.groupby(["archetype", "deck_signature", "is_winner_action"], dropna=False)
        .agg(
            action_rows=("action_trace_id", "count"),
            episodes=("episode_id", "nunique"),
            avg_reward=("reward", "mean"),
            avg_option_count=("option_count", "mean"),
            avg_turn=("turn", "mean"),
        )
        .reset_index()
    )
    save_csv(summary, DIRS["bc"] / "bc_action_trace_summary.csv")
    save_csv(pd.DataFrame([dict(total_stats)]), DIRS["bc"] / "action_trace_extraction_stats.csv")

    display_table(
        "v15: 勝率上位デッキの実リプレイ行動ログ summary",
        summary,
        "勝率上位デッキに紐づく実リプレイ行動ログの件数です。後段のBC学習では、`action_traces_top_decks_winner_only.parquet` を主に使います。",
        max_rows=100,
    )
    display_table(
        "v15: デッキ別 action trace ファイル索引",
        per_deck_index,
        "1行が1つのdeck_signatureです。`actions_path` がそのデッキの行動ログ保存先です。",
        max_rows=100,
    )
    return action_df, summary


action_traces_top_decks, action_trace_summary = collect_and_save_top_deck_action_traces(json_files, top_by_arch, deck_instances)


<a id="section-7"></a>

## 7. archetype相性

[目次へ戻る](#toc)

行を自分のarchetype、列を相手のarchetypeとして、対面勝率と対戦数を確認します。勝率だけでなく件数も見て、信頼できる相性かを判断します。


In [ ]:
def build_matchup_tables(matches):
    rows = []
    for eid, g in matches.groupby("episode_id"):
        g = g.dropna(subset=["archetype"]).copy()
        if len(g) < 2:
            continue
        for _, own in g.iterrows():
            for _, opp in g.iterrows():
                if own["player"] == opp["player"]:
                    continue
                rows.append({
                    "episode_id": str(eid),
                    "episode_date": own.get("episode_date"),
                    "own_player": own.get("player"),
                    "opponent_player": opp.get("player"),
                    "own_team": own.get("team"),
                    "opponent_team": opp.get("team"),
                    "own_archetype": own.get("archetype"),
                    "opponent_archetype": opp.get("archetype"),
                    "won": bool(own.get("won")),
                    "reward": own.get("reward", np.nan),
                    "final_prize_taken": own.get("final_prize_taken", np.nan),
                })

    perspective = pd.DataFrame(rows)
    if len(perspective) == 0:
        return perspective, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    summary = (
        perspective.groupby(["own_archetype", "opponent_archetype"], dropna=False)
        .agg(games=("episode_id", "count"), wins=("won", "sum"), avg_reward=("reward", "mean"), avg_prize_taken=("final_prize_taken", "mean"))
        .reset_index()
    )
    summary["winrate"] = summary["wins"] / summary["games"]
    summary["winrate_pct"] = (summary["winrate"] * 100).round(1)
    summary["display_priority"] = summary["games"] >= MIN_MATCHUP_GAMES_FOR_DISPLAY
    summary = summary.sort_values(["own_archetype", "display_priority", "winrate", "games"], ascending=[True, False, False, False])

    winrate_matrix = summary.pivot_table(index="own_archetype", columns="opponent_archetype", values="winrate", aggfunc="mean")
    games_matrix = summary.pivot_table(index="own_archetype", columns="opponent_archetype", values="games", aggfunc="sum").fillna(0).astype(int)
    return perspective, summary, winrate_matrix, games_matrix


matchup_perspective, matchup_summary, matchup_winrate_matrix, matchup_games_matrix = build_matchup_tables(matches)

save_csv(matchup_perspective, DIRS["analysis"] / "archetype_matchup_perspective.csv")
save_csv(matchup_summary, DIRS["analysis"] / "archetype_matchup_summary.csv")
save_csv(matchup_winrate_matrix, DIRS["analysis"] / "archetype_matchup_winrate_matrix.csv", index=True)
save_csv(matchup_games_matrix, DIRS["analysis"] / "archetype_matchup_games_matrix.csv", index=True)

display_table(
    "archetype相性表",
    matchup_summary,
    "自分のarchetypeと相手のarchetypeの組み合わせごとに、対戦数・勝数・勝率・平均rewardを集計した表です。`own_archetype` が自分、`opponent_archetype` が相手です。",
    max_rows=100,
)

display_table(
    "archetype相性: 勝率行列",
    matchup_winrate_matrix,
    "行が自分のarchetype、列が相手のarchetypeです。値は自分側の勝率です。",
)

display_table(
    "archetype相性: 対戦数行列",
    matchup_games_matrix,
    "行が自分のarchetype、列が相手のarchetypeです。値はその対面の対戦数です。勝率行列を見るときの信頼度確認に使います。",
)


def show_matchup_for_archetype(archetype_name, min_games=1, sort_by="winrate"):
    df = matchup_summary[matchup_summary["own_archetype"].astype(str) == str(archetype_name)].copy()
    df = df[df["games"] >= min_games]
    if sort_by == "games":
        df = df.sort_values(["games", "winrate"], ascending=[False, False])
    elif sort_by == "bad":
        df = df.sort_values(["winrate", "games"], ascending=[True, False])
    else:
        df = df.sort_values(["winrate", "games"], ascending=[False, False])
    return df[["own_archetype", "opponent_archetype", "games", "wins", "winrate_pct", "avg_reward", "avg_prize_taken"]]


def show_good_and_bad_matchups(archetype_name, min_games=5, n=5):
    display(Markdown(f"### {archetype_name}: 有利対面 top {n}"))
    display_table(
        f"{archetype_name}: 有利対面",
        show_matchup_for_archetype(archetype_name, min_games, "winrate").head(n),
        f"{archetype_name} 側から見て勝率が高い対面です。対象は `games >= {min_games}` の対面です。",
    )
    display(Markdown(f"### {archetype_name}: 不利対面 top {n}"))
    display_table(
        f"{archetype_name}: 不利対面",
        show_matchup_for_archetype(archetype_name, min_games, "bad").head(n),
        f"{archetype_name} 側から見て勝率が低い対面です。対象は `games >= {min_games}` の対面です。",
    )


for arch in tables["archetype_usage"]["archetype"].head(5).astype(str).tolist() if len(tables["archetype_usage"]) else []:
    show_good_and_bad_matchups(arch, min_games=MIN_MATCHUP_GAMES_FOR_DISPLAY, n=5)


<a id="section-8"></a>

## 8. 新archetype候補

[目次へ戻る](#toc)

`その他/不明` に残ったデッキから、汎用カードを除外し、勝率と採用カードの組み合わせを使って新archetype候補を探します。


In [ ]:
def candidate_label(row):
    if is_generic_candidate_card(row):
        return None
    name, stage = str(row.get("card_name", "")), str(row.get("jp_stage", ""))
    return final_evolution_name(name) if "ポケモン" in stage else name


def build_candidate_sets(decklists):
    rows = []
    unknown_decks = deck_instances[deck_instances["archetype"].astype(str).str.contains("その他|不明|Unknown", na=False)]
    unknown_list = decklists.merge(unknown_decks[["episode_id", "player"]], on=["episode_id", "player"], how="inner")

    for (eid, player), g in unknown_list.groupby(["episode_id", "player"]):
        labels = sorted(set(x for _, row in add_card_master_columns(g).iterrows() if (x := candidate_label(row))))
        rows.append({
            "episode_id": str(eid),
            "player": player,
            "won": bool(normalize_bool_series(g["won"]).iloc[0]) if "won" in g else None,
            "candidate_cards": labels,
        })
    return pd.DataFrame(rows)


def candidate_usage(candidate_sets, combo_size=1):
    counter, wins = Counter(), Counter()
    for r in candidate_sets.itertuples():
        cards = sorted(set(r.candidate_cards))
        combos = [(c,) for c in cards] if combo_size == 1 else itertools.combinations(cards[:25], combo_size)
        for combo in combos:
            combo = tuple(sorted(combo))
            counter[combo] += 1
            wins[combo] += int(bool(r.won))

    rows = []
    for combo, n in counter.items():
        if n < MIN_CANDIDATE_GAMES:
            continue
        wr = wins[combo] / n
        if wr >= MIN_CANDIDATE_WINRATE:
            rows.append({
                "candidate_type": f"combo_{combo_size}" if combo_size > 1 else "single",
                "candidate": " + ".join(combo),
                "player_decks": n,
                "wins": wins[combo],
                "winrate": wr,
            })
    return pd.DataFrame(rows).sort_values(["player_decks", "winrate"], ascending=[False, False]) if rows else pd.DataFrame()


candidate_sets = build_candidate_sets(decklists)
candidate_singles = candidate_usage(candidate_sets, 1)
candidate_pairs = candidate_usage(candidate_sets, 2)
candidate_triples = candidate_usage(candidate_sets, 3)
new_candidates = pd.concat([candidate_singles, candidate_pairs, candidate_triples], ignore_index=True) if any(len(x) for x in [candidate_singles, candidate_pairs, candidate_triples]) else pd.DataFrame()

save_csv(candidate_singles, DIRS["candidate"] / "candidate_singles_winrate_ge_50.csv")
save_csv(candidate_pairs, DIRS["candidate"] / "candidate_pairs_winrate_ge_50.csv")
save_csv(candidate_triples, DIRS["candidate"] / "candidate_triples_winrate_ge_50.csv")
save_csv(new_candidates, DIRS["candidate"] / "new_archetype_candidates_winrate_ge_50.csv")

display_table(
    "新archetype候補",
    new_candidates,
    f"`その他/不明` に残ったデッキから、汎用カードを除外し、勝率 `winrate >= {MIN_CANDIDATE_WINRATE}` かつ `player_decks >= {MIN_CANDIDATE_GAMES}` の候補を抽出した表です。",
    max_rows=100,
)


<a id="section-9"></a>

## 9. 可視化と出力一覧

[目次へ戻る](#toc)

主要なグラフを保存し、最後に保存したファイルの説明表を表示します。公開Notebookでは、生成物の意味が分かるように `output_file_descriptions.csv` も保存します。


In [ ]:
def plot_hbar(df, label_col, value_col, title, path, top_n=None, xlabel=None):
    if len(df) == 0:
        return
    d = df.head(top_n).copy() if top_n else df.copy()
    plt.figure(figsize=(10, max(5, 0.35 * len(d))))
    plt.barh(d[label_col].astype(str)[::-1], d[value_col][::-1])
    plt.xlabel(xlabel or value_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


def plot_archetype_winrates(archetype_usage):
    if len(archetype_usage) == 0:
        return
    df = archetype_usage[archetype_usage["player_decks"] >= MIN_ARCHETYPE_GAMES_FOR_WINRATE_PLOT].copy()
    df = df.sort_values(["winrate", "player_decks"], ascending=[True, True])
    plt.figure(figsize=(10, max(6, 0.35 * len(df))))
    plt.barh(df["archetype"].astype(str), df["winrate"] * 100)
    plt.xlim(0, 100)
    plt.xlabel("Winrate (%)")
    plt.title(f"Winrate by Archetype (player_decks >= {MIN_ARCHETYPE_GAMES_FOR_WINRATE_PLOT})")
    for i, r in enumerate(df.itertuples()):
        plt.text(min(float(r.winrate) * 100 + 1, 98), i, f"{float(r.winrate) * 100:.1f}% / n={int(r.player_decks)}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(DIRS["fig"] / "archetype_winrates.png", dpi=150)
    plt.show()


def plot_matchup_heatmap(winrate_matrix, games_matrix):
    if len(winrate_matrix) == 0:
        return
    order = tables["archetype_usage"]["archetype"].astype(str).tolist() if len(tables["archetype_usage"]) else list(winrate_matrix.index)
    rows = [a for a in order if a in winrate_matrix.index.astype(str).tolist()]
    cols = [a for a in order if a in winrate_matrix.columns.astype(str).tolist()]
    mat = winrate_matrix.loc[rows, cols]

    plt.figure(figsize=(max(9, 0.45 * len(cols)), max(7, 0.45 * len(rows))))
    im = plt.imshow(mat.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=1)
    plt.colorbar(im, label="Winrate")
    plt.xticks(range(len(cols)), cols, rotation=90)
    plt.yticks(range(len(rows)), rows)
    plt.xlabel("Opponent archetype")
    plt.ylabel("Own archetype")
    plt.title("Archetype matchup winrate matrix")

    for i, own in enumerate(rows):
        for j, opp in enumerate(cols):
            if pd.isna(mat.iloc[i, j]):
                continue
            games = int(games_matrix.loc[own, opp]) if own in games_matrix.index and opp in games_matrix.columns else 0
            if games >= MIN_MATCHUP_GAMES_FOR_DISPLAY:
                plt.text(j, i, f"{mat.iloc[i, j] * 100:.0f}%\n n={games}", ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(DIRS["fig"] / "archetype_matchup_heatmap.png", dpi=150)
    plt.show()


plot_hbar(tables["card_usage"], "card_name", "player_decks", "Top used cards", DIRS["fig"] / "top_cards.png", top_n=30, xlabel="Player-decks")
plot_hbar(tables["archetype_usage"], "archetype", "player_decks", "Top archetypes", DIRS["fig"] / "archetypes.png", top_n=30, xlabel="Player-decks")
plot_archetype_winrates(tables["archetype_usage"])
plot_matchup_heatmap(matchup_winrate_matrix, matchup_games_matrix)

def describe_output_file(path):
    """
    保存ファイルの用途を、人が読める説明に変換する。
    """
    rel = str(Path(path).relative_to(OUT_DIR))
    name = Path(path).name

    rules = [
        ("date_selection_summary.csv", "指定した日付・期間に対応するdaily replay datasetの存在確認表。json_countで各日付のJSON件数を確認できます。"),
        ("matches.csv", "対戦×プレイヤー単位の基本表。勝敗、archetype、最終ターン、取ったサイド数などを確認します。"),
        ("decklists.csv", "各プレイヤーの60枚デッキリスト。カードID、カード名、枚数、archetypeが入っています。"),
        ("event_logs.csv", "replay中のカード使用・進化・攻撃などのイベントログ。行動分析に使います。"),
        ("bad_files.csv", "読み込みに失敗したJSONファイル一覧。エラー確認用です。"),
        ("card_usage.csv", "カード別の採用数・合計採用枚数・採用デッキ勝率の表。環境で使われているカードを確認します。"),
        ("archetype_usage.csv", "archetype別の使用数・勝数・勝率の表。環境分布を見る中心的な出力です。"),
        ("strong_actions.csv", "カード使用・進化・攻撃など、行動単位の勝率表。強そうな行動候補の発見に使います。"),
        ("deck_instances.csv", "episode_id×playerを1つのデッキとしてまとめた表。deck_signatureで同一60枚デッキを識別します。"),
        ("exact_deck_performance_all.csv", "同一60枚デッキごとの対戦数・勝数・勝率の全件表。デッキ単位の性能を見る中心的な出力です。"),
        ("top10_decks_overall_min5games.csv", "全archetype横断で、対戦数5以上の勝率上位10デッキをまとめた表。"),
        ("top10_decks_overall_min5games.md", "全archetype横断の勝率上位10デッキを、人が読みやすいMarkdown形式で保存したもの。"),
        ("top5_decks_by_archetype_min5games.csv", "各archetypeについて、対戦数5以上の勝率上位1〜5位デッキをまとめた表。"),
        ("top5_decks_by_archetype_min5games.md", "各archetypeの勝率上位1〜5位デッキを、人が読みやすいMarkdown形式で保存したもの。"),
        ("top5_decklists_by_archetype_ready_to_use.csv", "各archetype上位デッキのカードリストを縦持ちにまとめた表。デッキ構築に再利用しやすい形式です。"),
        ("top_deck_files_index.csv", "ready_to_use_top_decks内の個別デッキファイルの索引。archetype、順位、勝率、ファイルパスを確認できます。"),
        ("archetype_matchup_perspective.csv", "archetype相性分析用のプレイヤー視点データ。自分archetype、相手archetype、勝敗が入っています。"),
        ("archetype_matchup_summary.csv", "自分archetype×相手archetypeごとの対戦数・勝数・勝率・平均rewardの表。相性確認の中心的な出力です。"),
        ("archetype_matchup_winrate_matrix.csv", "行が自分、列が相手のarchetype別勝率行列。ヒートマップの元データです。"),
        ("archetype_matchup_games_matrix.csv", "行が自分、列が相手のarchetype別対戦数行列。勝率の信頼度確認に使います。"),
        ("candidate_singles_winrate_ge_50.csv", "その他/不明デッキから抽出した単体カードの新archetype候補。勝率5割以上のみ。"),
        ("candidate_pairs_winrate_ge_50.csv", "その他/不明デッキから抽出した2カード組み合わせの新archetype候補。勝率5割以上のみ。"),
        ("candidate_triples_winrate_ge_50.csv", "その他/不明デッキから抽出した3カード組み合わせの新archetype候補。勝率5割以上のみ。"),
        ("new_archetype_candidates_winrate_ge_50.csv", "single/pair/tripleの新archetype候補をまとめた表。"),
        ("action_traces_top_decks_all.parquet", "勝率上位デッキに紐づく実リプレイ行動ログ全件。observation_json、legal_options_json、selected_indices_jsonを含み、BC/RL学習に使います。"),
        ("action_traces_top_decks_winner_only.parquet", "勝率上位デッキのうち勝者側だけの実リプレイ行動ログ。BC教師データの主入力です。"),
        ("bc_action_trace_summary.csv", "action traceの件数・episode数・平均合法手数などをarchetype/deck_signature別に集計した表。"),
        ("action_trace_extraction_stats.csv", "action trace抽出時の成功・失敗・skip理由の集計表。"),
        ("deck_action_files_index.csv", "デッキごとに保存したaction traceファイルの索引。deck_signatureとactions_pathを対応づけます。"),
        ("top_deck_files_index_with_actions.csv", "勝率上位デッキ索引にaction traceの件数と保存先を結合したもの。モデル学習Notebookの入力に便利です。"),
        ("top_cards.png", "カード採用数上位の棒グラフ。"),
        ("archetypes.png", "archetype使用数上位の棒グラフ。"),
        ("archetype_winrates.png", "archetype別勝率の棒グラフ。"),
        ("archetype_matchup_heatmap.png", "archetype相性の勝率ヒートマップ。行が自分、列が相手です。"),
        ("duplicate_card_name_master.csv", "同じ日本語名だがカードIDが複数あるカード一覧。効果違いカードの確認用です。"),
    ]

    for key, desc in rules:
        if name == key:
            return desc

    if "ready_to_use_top_decks" in rel and name.endswith(".csv"):
        return "各archetype上位デッキを1デッキ1ファイルで保存したCSV版。カードID・カード名・枚数を確認できます。"
    if "ready_to_use_top_decks" in rel and name.endswith(".txt"):
        return "各archetype上位デッキをコピーしやすいテキスト形式で保存したもの。"
    if "ready_to_use_top_decks" in rel and name.endswith(".py"):
        return "各archetype上位デッキをPythonで使いやすい `DECK_CARD_IDS` / `DECK_COUNTS` 形式で保存したもの。"

    if name.endswith(".csv"):
        return "CSV形式の分析結果です。"
    if name.endswith(".parquet"):
        return "Parquet形式の分析結果です。ネストした行動ログやobservationを保存するために使っています。"
    if name.endswith(".pkl"):
        return "pickle形式の分析結果です。Parquet保存に失敗した場合のfallbackです。"
    if name.endswith(".md"):
        return "Markdown形式のレポートです。"
    if name.endswith(".png"):
        return "グラフ画像です。"

    return "保存ファイルです。"


def build_output_file_description_table(out_dir=OUT_DIR):
    rows = []
    for p in sorted(Path(out_dir).rglob("*")):
        if not p.is_file():
            continue

        rows.append({
            "relative_path": str(p.relative_to(out_dir)),
            "file_name": p.name,
            "description": describe_output_file(p),
            "size_kb": round(p.stat().st_size / 1024, 1),
        })

    return pd.DataFrame(rows)


output_file_descriptions = build_output_file_description_table(OUT_DIR)
save_csv(output_file_descriptions, OUT_DIR / "output_file_descriptions.csv")

display_table(
    "output_file_descriptions: 保存したファイルの説明",
    output_file_descriptions,
    "このNotebookが保存したファイルの一覧です。relative_pathが保存場所、descriptionが用途です。最後に保存される `output_file_descriptions.csv` 自体にも、この一覧が入っています。",
    max_rows=200,
)
